In [9]:
# # Load required package
# library(rmarkdown)

# # List of your Rmd files
# rmd_files <- list.files(path = ".", pattern = "\\.Rmd$", full.names = TRUE)

# # Output directory
# output_dir <- "."

# # Create the output directory if it doesn't exist
# if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

# # Loop over each file
# for (rmd in rmd_files) {
#   render(
#     input = rmd,
#     output_format = "pdf_document",
#     output_dir = output_dir
#   )
# }

In [10]:
library(lme4)
library(lmerTest)
library(car)
library(ggplot2)

In [11]:
df <- read.csv("concatenated_data/experiment2_results.csv")
df$lang <- as.factor(df$lang)
df$preceding_tone <- as.factor(df$preceding_tone)
df$participant <- as.factor(df$participant)
df$word_freq_ratio <- scale(df$word_freq_ratio)

df$preceding_tone_offset <- as.integer(sub(".*(\\d)$", "\\1", df$preceding_tone))

df$level <- as.numeric(scale(df$level, center = TRUE, scale = FALSE))
df$preceding_tone_offset <- as.numeric(scale(df$preceding_tone_offset, center = TRUE, scale = FALSE))
df$fluency <- as.numeric(scale(df$fluency, center = TRUE, scale = FALSE))

cor(df$word_freq_ratio, df$falling_tone_chosen, method = "pearson")
cor(df$word_freq_ratio, df$preceding_tone_offset, method = "pearson")

0.03560648


0.1254859


In [12]:
temp <- df
contrasts(temp$preceding_tone) <- contr.sum(length(unique(temp$preceding_tone)))
print(contrasts(temp$preceding_tone))
model <- glmer(falling_tone_chosen ~ level*lang + preceding_tone_offset*lang + preceding_tone*lang + (1|participant), family=binomial, data=temp, control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e5)))
summary(model)

       [,1] [,2] [,3] [,4]
tone21    1    0    0    0
tone33    0    1    0    0
tone35    0    0    1    0
tone51    0    0    0    1
tone55   -1   -1   -1   -1


fixed-effect model matrix is rank deficient so dropping 8 columns / coefficients


Correlation matrix not shown by default, as p = 13 > 12.
Use print(obj, correlation=TRUE)  or
    vcov(obj)        if you need it




Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: falling_tone_chosen ~ level * lang + preceding_tone_offset *  
    lang + preceding_tone * lang + (1 | participant)
   Data: temp
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e+05))

     AIC      BIC   logLik deviance df.resid 
  5282.9   5386.4  -2627.4   5254.9    12036 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-179.02   -0.15   -0.01    0.15 1453.68 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.2404   0.4903  
Number of obs: 12050, groups:  participant, 43

Fixed effects:
                                        Estimate Std. Error z value Pr(>|z|)
(Intercept)                             -2.18954    0.18368 -11.920  < 2e-16
level                                    2.04406    0.10864  18.814  < 2e-16
langman_bilingual                        0.85401    0.24324   3.511 0.

In [13]:
# Which fixed-effect columns were dropped?
dropped <- attr(getME(model, "X"), "col.dropped")

dropped

preceding_tone4           langmin:preceding_tone1 
                                9                                15 
langman_bilingual:preceding_tone2           langmin:preceding_tone2 
                               16                                17 
langman_bilingual:preceding_tone3           langmin:preceding_tone3 
                               18                                19 
langman_bilingual:preceding_tone4           langmin:preceding_tone4 
                               20                                21

In [14]:
temp$residual <- residuals(model)
model <- lmer(residual ~ word_freq_ratio + (1|participant), data=temp)
summary(model) # Post-hoc residual analysis to see if word frequency ratio is a significant predictor of the remaining variance

boundary (singular) fit: see help('isSingular')



Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: residual ~ word_freq_ratio + (1 | participant)
   Data: temp

REML criterion at convergence: 23973.4

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-6.9491 -0.3224 -0.0082  0.3394  8.2644 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.0000   0.0000  
 Residual                0.4276   0.6539  
Number of obs: 12050, groups:  participant, 43

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)
(Intercept)     -6.512e-03  5.957e-03  1.205e+04  -1.093    0.274
word_freq_ratio  4.175e-03  5.957e-03  1.205e+04   0.701    0.483

Correlation of Fixed Effects:
            (Intr)
word_frq_rt 0.000 
optimizer (nloptwrap) convergence code: 0 (OK)
boundary (singular) fit: see help('isSingular')


In [15]:
temp <- df
temp <- temp[(temp$lang =="min") | (temp$lang =="man_bilingual"),]
temp$lang <- droplevels(temp$lang)
contrasts(temp$preceding_tone) <- contr.sum(length(unique(temp$preceding_tone)))
print(contrasts(temp$lang))
print(contrasts(temp$preceding_tone))
model <- glmer(falling_tone_chosen ~ level*lang + preceding_tone_offset*lang + preceding_tone*lang + (level|participant), family=binomial, data=temp, control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e5)))
summary(model)

              min
man_bilingual   0
min             1
       [,1] [,2] [,3] [,4]
tone21    1    0    0    0
tone33    0    1    0    0
tone35    0    0    1    0
tone51    0    0    0    1
tone55   -1   -1   -1   -1


fixed-effect model matrix is rank deficient so dropping 5 columns / coefficients



Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: falling_tone_chosen ~ level * lang + preceding_tone_offset *  
    lang + preceding_tone * lang + (level | participant)
   Data: temp
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e+05))

     AIC      BIC   logLik deviance df.resid 
  4444.6   4530.9  -2210.3   4420.6     9788 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-121.791   -0.165   -0.011    0.168   97.548 

Random effects:
 Groups      Name        Variance Std.Dev. Corr
 participant (Intercept) 0.21162  0.4600       
             level       0.09104  0.3017   0.04
Number of obs: 9800, groups:  participant, 28

Fixed effects:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                   -1.61234    0.12511 -12.887  < 2e-16 ***
level                          1.90916    0.08822  21.641  < 2e-16 ***
langmin          

In [17]:
# Discussion on bilingualism: Does fluency in TSM affect perceptual compensation in TM?
temp <- df
temp <- temp[(temp$lang =="man_bilingual"),]
temp$preceding_tone <- droplevels(temp$preceding_tone)
temp$lang <- droplevels(temp$lang)
contrasts(temp$preceding_tone) <- contr.sum(length(unique(temp$preceding_tone)))
print(contrasts(temp$preceding_tone))
model <- glmer(falling_tone_chosen ~ level*fluency + preceding_tone_offset*fluency + preceding_tone*fluency + (1|participant), family=binomial, data=temp, control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e5)))
summary(model)

       [,1] [,2]
tone35    1    0
tone51    0    1
tone55   -1   -1


fixed-effect model matrix is rank deficient so dropping 2 columns / coefficients



Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: falling_tone_chosen ~ level * fluency + preceding_tone_offset *  
    fluency + preceding_tone * fluency + (1 | participant)
   Data: temp
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e+05))

     AIC      BIC   logLik deviance df.resid 
  1477.4   1534.5   -729.7   1459.4     4191 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-271.350   -0.088   -0.005    0.087  123.356 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.8345   0.9135  
Number of obs: 4200, groups:  participant, 28

Fixed effects:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                   -0.83233    0.23366  -3.562 0.000368 ***
level                          1.94315    0.09027  21.526  < 2e-16 ***
fluency                       -0.08409    0.10858  -0.774 0.438686    

In [19]:
temp <- df
temp <- temp[(temp$lang =="min"),]
temp$lang <- droplevels(temp$lang)
model <- glmer(falling_tone_chosen ~ level + fluency + preceding_tone_offset*fluency + (1|participant), family=binomial, data=temp, control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e5)))
summary(model)

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: falling_tone_chosen ~ level + fluency + preceding_tone_offset *  
    fluency + (1 | participant)
   Data: temp
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e+05))

     AIC      BIC   logLik deviance df.resid 
  3366.9   3406.7  -1677.5   3354.9     5594 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-11.8434  -0.2621  -0.0667   0.2857  19.1559 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.135    0.3674  
Number of obs: 5600, groups:  participant, 28

Fixed effects:
                               Estimate Std. Error z value Pr(>|z|)    
(Intercept)                   -0.450333   0.104442  -4.312 1.62e-05 ***
level                          1.086016   0.028198  38.514  < 2e-16 ***
fluency                       -0.001144   0.047912  -0.024    0.981    
preceding_tone_offset 